# Text clip finetune example


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

# Add path to more easily load checkpoints of different architectures
model_utils_path = ""  # TODO: PATH_UPDATE fine-tuning codebase path

import torch
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook

In [ ]:
torch_precision = torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
lr = 0.05
n_iter = 500

In [ ]:
global_seed = 3 # Can be None

In [ ]:
import open_clip
from utils.train import train_with_criterion
from torchvision.transforms.v2 import RandomChoice, RandomPerspective, RandomResizedCrop, RandomHorizontalFlip, GaussianBlur, Identity, Transform
from utils.losses.clip import CLIPDirectionalCosineSimilarity

clip_model_name = 'ViT-B-16-SigLIP-512'
clip_pretrained = 'webli'

from scenes import SpringScene, SciFiRobotScene, CarScene, BlenderManScene, HouseScene, DinoScene, FlowerPotScene, RedCarScene, EinarSmallDomeScene, SpringPortraitSmallDomeScene
scene = SciFiRobotScene()
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

models_to_use = [""]  # TODO: PATH_UPDATE fine-tuned checkpoint path
for fine_tune in models_to_use:

    model, _, preprocess_eval = open_clip.create_model_and_transforms(clip_model_name, pretrained=clip_pretrained, device=device)
    tokenizer = open_clip.get_tokenizer(clip_model_name)
    
    
    special_architecture=True
    if fine_tune and not special_architecture:
        model.load_state_dict(torch.load(fine_tune, map_location=device))
        print("Loaded fine-tuned weights from ", fine_tune)
    elif fine_tune and special_architecture:
        # Load fine-tuned model
        model_path = fine_tune
        from load_checkpoints import load_model_from_checkpoint_path
        clip_model, tokenizer, clip_preprocess = load_model_from_checkpoint_path(model_path)
        clip_model.eval()
        print(f"Loaded fine-tuned CLIP model from: {model_path}")
    model.eval()
    print("Model loaded! :)")
    
    prompts = [
        # lighting description distribution
        ('boring, ugly, even', 'stunningly beautiful lighting')
    ]

    for initial_prompt, target_prompt in prompts:
    
        criterion = CLIPDirectionalCosineSimilarity(initial_prompt, target_prompt, scene.get_combined_image(color_space_converter).permute(2, 1, 0), model, tokenizer, device=device, preprocess=preprocess_eval, always_prenormalize_vectors=True)
        # from utils.losses.image_image import ImageImageCLIPLoss
        title_prefix = "ImageTextFineTuning Comparison"

        size = model.visual.preprocess_cfg['size'] or (224, 224)
        
        train_with_criterion(
            scene,
            lr, n_iter, criterion,
            starting_multiplier_std=(0.3, 0.3, 0.3),
            output_subdirectory_name="text_clip_finetune_example",
            n_results=4,
            torch_precision=torch_precision,
            # augmentation=RandomChoice([RandomResizedCrop(size=size, scale=(0.1, 1.0), antialias=True)]),
            render_color_space_converter=color_space_converter,
            require_physically_plausible_multipliers=True,
            title_prefix=title_prefix + "Fine-Tuned Model" if fine_tune else "Original Model",
            device=device,
            save_every=40,
            model_name=clip_model_name,
            pretrained_source=fine_tune,
            seed=global_seed,
            show_images_after_augmentation=False
        )